# Librerias necesarias para el preprocesamiento de datos


In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report, ConfusionMatrixDisplay

## Carga del dataset local e información útil de los datos
Se carga el dataset de manera local y se consulta los tipos de datos de cada característica del dataset

*El Dataset se encuentra sesgado 4 a 1 en las clases objetivo*

In [ ]:
telcoCustomers = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')

In [ ]:
telcoCustomers.info()
print(telcoCustomers.Churn.value_counts())

##  Valores nulos por caracteristica en el dataset

Aunque en la sumatoria pueda mostrar que no haya valores nulos/NaN hay que verificar el tipo de dato de cada caracteristica del dataframe para poder tratar datos numericos identificados como objetos haciendo engañoso a un NaN como un espacio en blanco

In [ ]:
print(telcoCustomers.isna().sum())

In [ ]:
df = telcoCustomers.copy()

In [ ]:

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors= 'coerce')
df['MonthlyCharges'] = pd.to_numeric(df['MonthlyCharges'], errors= 'coerce')

print('total de datos nulos:\n', df[['TotalCharges', 'MonthlyCharges']].isnull().sum())
df.dropna(inplace=True)

# dfTargetYes= df[df['Churn']=='Yes']
# dfTargetNo= df[df['Churn']=='No']
# dfChurnNo = dfTargetNo.sample(len(dfTargetYes), random_state=1)
# dfBalanceado = pd.concat([dfChurnNo, dfTargetYes]) 
# target = dfBalanceado['Churn']
# data = dfBalanceado.drop(columns=['Churn', 'customerID'])

target = df['Churn']
data = df.drop(columns=['Churn', 'customerID'])

In [ ]:
print(data.isna().sum())

In [ ]:

data['MultipleLines'] = data['MultipleLines'].replace('No phone service', 'No')    

columnasInternet = ['StreamingMovies', 'StreamingTV', 'TechSupport', 'DeviceProtection', 'OnlineBackup', 'OnlineSecurity',]

for col in columnasInternet:
    data[col]= data[col].replace('No internet service', 'No')

for col in data.select_dtypes(include='object').columns:
    if col != 'customerID':
        print(f"--- Columna: {col} ---")
        print(data[col].value_counts(dropna=False).head(5))
        print("\n")


## Identificar el tipo de codificación para cada columna del dataset
Cada tipo de dato ya sea numérico o categórico requiere una codificación especifica para el modelo de clasificación 

In [ ]:
data.info()

In [ ]:
caracteristicasNumericas = ['tenure', 'MonthlyCharges', 'TotalCharges']
caracteristicasNominales = ['PaymentMethod', 'InternetService']
caracteristicasBinarias= ['gender' ,'SeniorCitizen' , 'Partner' ,'StreamingMovies', 
                                'StreamingTV', 'TechSupport', 'DeviceProtection', 'OnlineBackup', 
                                'OnlineSecurity' ,'Dependents', 'PhoneService', 'MultipleLines', 'PaperlessBilling']
caracteristicasOrdinales =['Contract']

contractOrden = [['Month-to-month', 'One year', 'Two year']]

## Imputer y Column Transformer con Pipelines
No se hace uso de imputers debido a que se revisó previamente que las categorías no contuvieran cadenas vacias ni que los datos numéricos fueran NaN

In [ ]:
transformadorNumerico = Pipeline(
    steps=[
        # ('imputer', SimpleImputer(strategy= 'median')),
        ('scalar', StandardScaler())        
    ]
)

In [ ]:
transformadorOrdinal = Pipeline(
    steps=[
        ('ordinal', OrdinalEncoder(categories=contractOrden))         
    ]
)

In [ ]:
transformadorNominal = Pipeline(
    steps=[
        ('oneHot', OneHotEncoder(handle_unknown='ignore'))
    ]
)

In [ ]:
transformadorBinario= Pipeline(
        steps=[
            ('binario', OrdinalEncoder())
        ]
    )

## Preprocesamiento de los datos
Se realiza un preprocesamiento de los datos con un pipeline y posteriormente se pasa al modelo el dataset limpio y normalizado


In [ ]:
preprocesador = ColumnTransformer(
    transformers=[
        ('binario', transformadorBinario, caracteristicasBinarias),
        ('numerico', transformadorNumerico, caracteristicasNumericas),
        ('ordinal', transformadorOrdinal, caracteristicasOrdinales), 
        ('nominal', transformadorNominal, caracteristicasNominales)
    ]
)

## Modelos

El árbol de desición de sklearn solo permite implementar árboles binarios con distintos modos de hallar el nodo raíz y las ramas, usando entropía o gini.

In [ ]:
arbolDesicion = DecisionTreeClassifier(random_state=1,criterion='entropy', class_weight='balanced', max_depth=3)

In [ ]:
pipeline = Pipeline(
    steps=[
        ('preprocesador', preprocesador),('arbol', arbolDesicion)
    ]
)
Xtrain, Xtest, Ytrain, Ytest = train_test_split(data, target, random_state=42 , test_size=0.20)
pipeline.fit(Xtrain, Ytrain)

yPred= pipeline.predict(Xtest)

In [ ]:
nombreColumnas = pipeline.named_steps['preprocesador'].get_feature_names_out()
plt.figure(figsize=(20,10))
plot_tree(
    pipeline.named_steps['arbol'], 
    feature_names= nombreColumnas, class_names=['No', 'Yes'],
    filled=True, max_depth=3) 
plt.show()

## Explicación

Contract son 3 de manera ordinal y su entropia es 1, los separó y combinó para ver cuál separaba mejor en base al menor sesgo que se puede ver con la entropía, luego se busco al grupo cuales eran los dos con mayor ganancia de informacion o los que mejor separan la información del total que les da Contract y lo mismo con sus hijos pero ya no usando el dataset global si no el que va dejando el padre.



## Evaluación del modelo

Se hace uso de métricas para medir la eficiencia del modelo para clasificar clientes que van a cancelar de aquellos que no para tomar desición estrategica con ellos. 

In [ ]:
print(f"Accuracy {accuracy_score(Ytest, yPred)}: ")

In [ ]:
user = data.iloc[[4]] 

prediccion = pipeline.predict(user)[0]

print(f"Predicción: {prediccion}")

In [ ]:
cm = confusion_matrix(Ytest, yPred)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=pipeline.classes_)
disp.plot(cmap=plt.cm.Blues)

plt.title("Matriz de Confusión - Predicción de Churn")
plt.show()

## Métricas

 * La presición nos dice qué porcentaje de individuos acertó con una clase 
 * El recall nos dice qué porcentaje el porcentaje de aciertos o verdaderos positivos por clase
 * El f1-score es la media armónica de presición y recall 

In [ ]:
print(classification_report(Ytest, yPred))